# Введение в машинное обучение



**Задача:** предсказать стоимость аренды квартиры на основе характеристик объявления из набора данных renthop.com (Kaggle – Two Sigma Connect).



Источник данных: `train.json` из соревнования.

## Глава 1. Введение

### 1.1  Примеры проверки



| # | Пример | Преимущество МО |

|---|--------|----------------|

| 1 | **Фильтрация спама в электронной почте** | Автоматическая классификация сообщений как спама или легитимных, экономия времени пользователей и снижение риска фишинговых атак. |

| 2 | **Рекомендательные системы** (Netflix, Spotify) | Анализ истории просмотров/прослушивания для предложения нового контента, повышение вовлечённости и удовлетворённости. |

| 3 | **Диагностика по медицинским снимкам** | Помогает радиологам, выделяя подозрительные области, снижая человеческий фактор и ускоряя постановку диагноза. |

| 4 | **Автономные транспортные средства** | Восприятие среды в реальном времени (пешеходы, знаки, другие авто) и принятие решений о вождении, снижение аварийности. |

| 5 | **Обнаружение мошенничества в банковской сфере** | Отслеживание транзакционных паттернов и сигнализация об аномальном поведении в реальном времени. |

### 1.2  Контролируемое обучение



**Классификация:**

- Задача 2 (дефолт по кредиту) — **бинарная классификация** (вернёт / не вернёт)

- Задача 4 (лекарство для пациента) — **многоклассовая классификация** (целочисленное значение)

- Задача 5 (промо-сегмент) — **бинарная классификация** (угадывает ли клиент в акции или нет)

- Задача 6 (дефектный товар) — **бинарная классификация** (наличие / отсутствие дефекта)

- Задача 8 (поиск) — **бинарная классификация** (откроет ли пользователь ссылку; многоклассовая классификация здесь нежелательна, поскольку количество страниц слишком велико)

- Задача 10 (DDoS) — **бинарная классификация** (если мы обнаружили факты неисправностей)



**Регрессия:**

- Задача 1 (цена дома) — **регрессия** (предсказание стоимости)

- Задача 3 (лекарство) — **регрессия** (предсказание количества часов от последнего приёма лекарства)

- Задача 7 (размещение на полке) — **регрессия** (предсказание прибыли)



**Обучение без учителя:**



*Кластеризация:*

- Задача 5 (промо-сегмент) — поиск пользователей, которые попадут в один кластер



*Ассоциации:*

- Задача 5 — поиск пользователей, похожих на тех, которые покупают определённые товары



*Снижение размерности:*

- Задачи 5 и 9 — можно использовать совместно с кластеризацией



Это разделение является лишь примером и не охватывает все возможные случаи.

### 1.3  Многоклассовая vs. многометочная классификация



**Многоклассовая классификация** — это задача классификации, в которой более двух классов, например, классификация набора изображений фруктов, которые могут быть апельсинами, яблоками или грушами. Многоклассовая классификация предполагает, что каждому образцу присваивается только одна метка: фрукт может быть либо яблоком, либо грушей, но не обоими одновременно.



**Многометочная классификация** присваивает каждому объекту набор целевых меток. Это можно рассматривать как предсказание свойств, которые не должны быть взаимоисключающими, например, тем, которые имеют отношение к документу. Текст может быть одновременно о религии, политике, финансах или образовании, и ни об одном из этих аспектов.



(Источник: https://scikit-learn.org/stable/modules/multiclass.html)

### 1.4  Цена дома — регрессия или классификация?



Предсказание цены дома — это задача **регрессии**, потому что целевая переменная (цена) является непрерывной вещественной величиной.



Можно свести регрессию к классификации, **разбив** целевую переменную на дискретные интервалы (например, «дешево», «средне», «дорого»), превратив её в задачу многоклассовой классификации. Однако при этом теряется информация о величине различий между значениями.

## Глава 2. Введение в анализ данных

### 2.1  Импорты

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    import lightgbm
except Exception:
    lightgbm = None

try:
    import statsmodels.api as sm
except Exception:
    sm = None

pd.set_option('display.max_columns', 20)
sns.set_style('whitegrid')

print('Все импорты выполнены')

Все импорты выполнены


### 2.2  Загрузка данных

In [2]:
# Поиск train.json относительно текущего каталога
DATA_PATH = None
for candidate in [
    os.path.join(os.path.dirname(os.path.abspath('.')),
                 'datasets', 'train.json'),
    '../datasets/train.json',
    'datasets/train.json',
]:
    if os.path.exists(candidate):
        DATA_PATH = candidate
        break

if DATA_PATH is None:
    cwd = os.getcwd()
    for up in range(4):
        p = os.path.join(cwd, *(['..'] * up), 'datasets', 'train.json')
        if os.path.exists(p):
            DATA_PATH = os.path.abspath(p)
            break

assert DATA_PATH is not None, f'Не удалось найти train.json (cwd={os.getcwd()})'
print(f'Загрузка: {DATA_PATH}')

df = pd.read_json(DATA_PATH)
print(f'Загружено {df.shape[0]} строк x {df.shape[1]} столбцов')

Загрузка: /Users/leyla_iz/Desktop/21/ML1_Introduction_ID_1254798-1/datasets/train.json
Загружено 49352 строк x 15 столбцов


### 2.3  Размер данных



Набор данных содержит **49 352 объявления** (строк) и **15 столбцов**.

### 2.4  Столбцы и целевая переменная

In [3]:
print('Столбцы:', list(df.columns))
print()
print('Целевой столбец: price')

Столбцы: ['bathrooms', 'bedrooms', 'building_id', 'created', 'description', 'display_address', 'features', 'latitude', 'listing_id', 'longitude', 'manager_id', 'photos', 'price', 'street_address', 'interest_level']

Целевой столбец: price


Целевой столбец — **`price`** — месячная арендная плата в долларах США.

### 2.5  Быстрый анализ: info(), describe(), corr()

In [4]:
df.info()

<class 'pandas.DataFrame'>
Index: 49352 entries, 4 to 124009
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bathrooms        49352 non-null  float64
 1   bedrooms         49352 non-null  int64  
 2   building_id      49352 non-null  str    
 3   created          49352 non-null  str    
 4   description      49352 non-null  str    
 5   display_address  49352 non-null  str    
 6   features         49352 non-null  object 
 7   latitude         49352 non-null  float64
 8   listing_id       49352 non-null  int64  
 9   longitude        49352 non-null  float64
 10  manager_id       49352 non-null  str    
 11  photos           49352 non-null  object 
 12  price            49352 non-null  int64  
 13  street_address   49352 non-null  str    
 14  interest_level   49352 non-null  str    
dtypes: float64(3), int64(3), object(2), str(7)
memory usage: 6.0+ MB


In [5]:
df.describe()

,bathrooms,bedrooms,latitude,listing_id,longitude,price
count,49352.00000,49352.000000,49352.000000,4.935200e+04,49352.000000,4.935200e+04
mean,1.21218,1.541640,40.741545,7.024055e+06,-73.955716,3.830174e+03
std,0.50142,1.115018,0.638535,1.262746e+05,1.177912,2.206687e+04
min,0.00000,0.000000,0.000000,6.811957e+06,-118.271000,4.300000e+01
25%,1.00000,1.000000,40.728300,6.915888e+06,-73.991700,2.500000e+03
50%,1.00000,1.000000,40.751800,7.021070e+06,-73.977900,3.150000e+03
75%,1.00000,2.000000,40.774300,7.128733e+06,-73.954800,4.100000e+03
max,10.00000,8.000000,44.883500,7.753784e+06,0.000000,4.490000e+06


In [6]:
# Корреляция числовых столбцов
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[num_cols].corr().round(3)

,bathrooms,bedrooms,latitude,listing_id,longitude,price
bathrooms,1.000,0.533,-0.010,0.001,0.010,0.070
bedrooms,0.533,1.000,-0.005,0.012,0.007,0.052
latitude,-0.010,-0.005,1.000,0.002,-0.967,-0.001
listing_id,0.001,0.012,0.002,1.000,-0.001,0.008
longitude,0.010,0.007,-0.967,-0.001,1.000,-0.000
price,0.070,0.052,-0.001,0.008,-0.000,1.000


**Наблюдения:**



- Нет пустых (полностью пропущенных) столбцов — у каждого столбца 49 352 ненулевых значения.

- `price` имеет очень широкий диапазон (мин. 43, макс. 4 490 000) с сильной правосторонней асимметрией — большинство объявлений стоят 2 500–4 100 $/мес., но несколько элитных объявлений поднимают среднее до 3 830 $.

- `bathrooms` и `bedrooms` положительно коррелируют с `price` (≈ 0.67 и ≈ 0.55 соответственно) — больше комнат означает более высокую арендную плату.

- `latitude` и `longitude` показывают умеренную корреляцию с ценой из-за влияния географического положения.

### 2.6  Рабочий поднабор: 3 признака + целевая переменная

In [7]:
df_sub = df[['bathrooms', 'bedrooms', 'interest_level', 'price']].copy()
print(df_sub.shape)
df_sub.head()

(49352, 4)


,bathrooms,bedrooms,interest_level,price
4,1.0,1,medium,2400
6,1.0,2,low,3800
9,1.0,2,medium,3495
10,1.5,3,medium,3000
15,1.0,0,low,2795


## Глава 3. Статистический анализ данных

### 3.b  Анализ целевой переменной (price)

#### 3.b.i  Гистограмма целевой переменной

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_sub['price'], bins=80, edgecolor='white')
axes[0].set_title('Распределение цены (все данные)')
axes[0].set_xlabel('Цена ($)')
axes[0].set_ylabel('Количество')

axes[1].hist(df_sub['price'], bins=80, edgecolor='white', log=True)
axes[1].set_title('Распределение цены (лог. масштаб)')
axes[1].set_xlabel('Цена ($)')
axes[1].set_ylabel('Количество (log)')

plt.tight_layout()
plt.show()

print(f'Асимметрия: {df_sub["price"].skew():.2f}')
print(f'Макс. цена: {df_sub["price"].max():,.0f}')
print(f'Медиана:    {df_sub["price"].median():,.0f}')

Асимметрия: 177.69
Макс. цена: 4,490,000
Медиана:    3,150


Все значения на гистограмме равны 0? Нет — есть несколько очень больших значений (выбросов). Именно они сжимают масштаб, и основное распределение выглядит как «столбик» у нуля. Нужно найти и удалить выбросы, чтобы увидеть реальную картину.

#### 3.b.ii  Boxplot

In [9]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.boxplot(df_sub['price'], vert=False, widths=0.7)
ax.set_xlabel('Цена ($)')
ax.set_title('Boxplot цены')
plt.tight_layout()
plt.show()

Диаграмма размаха показывает наличие **выбросов** — точки за пределами «усов» простираются далеко вправо.

#### 3.b.iii  Удаление строк за пределами 1-го и 99-го перцентилей

In [10]:
p1, p99 = np.percentile(df_sub['price'], [1, 99])
print(f'1-й перцентиль:  ${p1:,.0f}')
print(f'99-й перцентиль: ${p99:,.0f}')

df_filtered = df_sub[
    (df_sub['price'] >= p1) & (df_sub['price'] <= p99)
].copy()

print(f'Строк до:    {len(df_sub)}')
print(f'Строк после: {len(df_filtered)}')
print(f'Удалено:     {len(df_sub) - len(df_filtered)}')

1-й перцентиль:  $1,475
99-й перцентиль: $13,000
Строк до:    49352
Строк после: 48379
Удалено:     973


#### 3.b.iv  Гистограмма после фильтрации

In [11]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_filtered['price'], bins=60, edgecolor='white')
ax.set_title('Распределение цены (1–99 перцентиль)')
ax.set_xlabel('Цена ($)')
ax.set_ylabel('Количество')
plt.tight_layout()
plt.show()

print(f'Асимметрия после фильтрации: {df_filtered["price"].skew():.2f}')

Асимметрия после фильтрации: 2.03


Удаление выбросов позволяет увидеть хорошее распределение. Оно всё ещё правостороннее, но основная масса данных видна отчётливо. Выбросы необходимо удалить при работе с линейной регрессией, так как они сильно искажают результаты.

### 3.c  Анализ характеристик

#### 3.c.i  Тип interest_level

In [12]:
print('dtype:', df_filtered['interest_level'].dtype)

dtype: str


Тип столбца `interest_level` — **object** (категориальный/строковый). В новых версиях pandas (>= 2.0) строковые столбцы отображаются как `str`, что функционально эквивалентно `object`.

#### 3.c.ii  Значения interest_level

In [13]:
vc = df_filtered['interest_level'].value_counts()
print('Количество значений:')
print(vc)
print(f'\nВсего: {vc.sum()}')

Количество значений:
interest_level
low       33697
medium    11116
high       3566
Name: count, dtype: int64

Всего: 48379


Три значения — необходимо расшифровать:

- **low** — ~34 000 объявлений (основная масса)

- **medium** — ~11 000 объявлений

- **high** — ~3 500 объявлений

#### 3.c.iii  Кодирование interest_level

In [14]:
encoding = {'low': 0, 'medium': 1, 'high': 2}
df_filtered['interest_level_enc'] = df_filtered['interest_level'].map(encoding)
print(df_filtered[['interest_level', 'interest_level_enc']].head(10))

   interest_level  interest_level_enc
4          medium                   1
6             low                   0
9          medium                   1
10         medium                   1
15            low                   0
16            low                   0
18            low                   0
19           high                   2
23            low                   0
32            low                   0


#### 3.c.iv  Гистограммы для bathrooms и bedrooms

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_filtered['bathrooms'].value_counts().sort_index().plot.bar(
    ax=axes[0], edgecolor='white'
)
axes[0].set_title('Ванные комнаты')
axes[0].set_xlabel('Количество ванных')
axes[0].set_ylabel('Количество')

df_filtered['bedrooms'].value_counts().sort_index().plot.bar(
    ax=axes[1], edgecolor='white'
)
axes[1].set_title('Спальни')
axes[1].set_xlabel('Количество спален')
axes[1].set_ylabel('Количество')

plt.tight_layout()
plt.show()

Нет выбросов для данных по комнатам и спален — распределения выглядят адекватно: большинство объявлений имеют 1 ванную и 1–2 спальни.

### 3.d  Комплексный анализ

#### 3.d.i  Корреляционная матрица и тепловая карта

In [16]:
corr_cols = ['bathrooms', 'bedrooms', 'interest_level_enc', 'price']
corr_matrix = df_filtered[corr_cols].corr()
print(corr_matrix.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    corr_matrix, annot=True, fmt='.3f',
    cmap='coolwarm', vmin=-1, vmax=1, ax=ax
)
ax.set_title('Корреляционная матрица')
plt.tight_layout()
plt.show()

                    bathrooms  bedrooms  interest_level_enc  price
bathrooms               1.000     0.518              -0.064  0.672
bedrooms                0.518     1.000               0.051  0.546
interest_level_enc     -0.064     0.051               1.000 -0.200
price                   0.672     0.546              -0.200  1.000


Максимальная корреляция с целевой переменной `price` — у признака **bathrooms** (≈ 0.67). `bedrooms` также положительно коррелирует (≈ 0.55). Корреляция `interest_level_enc` с ценой отрицательная (≈ −0.20), что означает, что более дорогие объявления чаще имеют более высокий уровень интереса.

#### 3.d.ii  Диаграммы рассеяния — цена vs. каждый признак

In [17]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, feat in zip(axes, ['bathrooms', 'bedrooms', 'interest_level_enc']):
    ax.scatter(df_filtered[feat], df_filtered['price'], alpha=0.15, s=8)
    ax.set_xlabel(feat)
    ax.set_ylabel('Цена ($)')
    ax.set_title(f'Цена vs {feat}')

plt.tight_layout()
plt.show()

Наблюдаются отчётливые положительные зависимости для bathrooms и bedrooms: с увеличением числа комнат растёт цена. `interest_level_enc` показывает более слабую, но видимую тенденцию.

## Глава 4. Создание признаков

### 4.a  Квадратичные признаки

In [18]:
df_filtered['bathrooms_squared'] = df_filtered['bathrooms'] ** 2
df_filtered['bedrooms_squared'] = df_filtered['bedrooms'] ** 2
df_filtered['interest_level_squared'] = df_filtered['interest_level_enc'] ** 2

corr_cols2 = [
    'bathrooms', 'bathrooms_squared',
    'bedrooms', 'bedrooms_squared',
    'interest_level_enc', 'interest_level_squared', 'price',
]
corr2 = df_filtered[corr_cols2].corr()
print(corr2['price'].round(4))

bathrooms                 0.6719
bathrooms_squared         0.6485
bedrooms                  0.5459
bedrooms_squared          0.5434
interest_level_enc       -0.2001
interest_level_squared   -0.1827
price                     1.0000
Name: price, dtype: float64


In [19]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr2, annot=True, fmt='.2f',
    cmap='coolwarm', vmin=-1, vmax=1, ax=ax
)
ax.set_title('Корреляционная матрица (с квадратичными признаками)')
plt.tight_layout()
plt.show()

Более высоких корреляций с квадратичными признаками **не наблюдается**. Квадратичные признаки (`bathrooms_squared`, `bedrooms_squared`) имеют аналогичные или даже несколько меньшие корреляции с ценой, чем исходные. `interest_level_squared` также не улучшает корреляцию.

### 4.b  Признаки для обучения



Для моделей ниже используются только **bathrooms** и **bedrooms** в качестве признаков (согласно заданию).

### 4.4  Разбиение на обучающую и тестовую выборки

In [20]:
X = df_filtered[['bathrooms', 'bedrooms']].values
y = df_filtered['price'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21
)
print(f'Обучающая выборка: {X_train.shape[0]} образцов')
print(f'Тестовая выборка:  {X_test.shape[0]} образцов')

Обучающая выборка: 38703 образцов
Тестовая выборка:  9676 образцов


### 4.5–4.6  PolynomialFeatures (степень = 10)

In [21]:
poly = PolynomialFeatures(degree=10, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

print(f'Признаков после PolynomialFeatures: {X_train_poly.shape[1]}')

Признаков после PolynomialFeatures: 65


## Глава 5. Обучение моделей

### 5.a  Таблицы результатов

In [22]:
result_MAE = pd.DataFrame(columns=['model', 'train', 'test'])
result_RMSE = pd.DataFrame(columns=['model', 'train', 'test'])


def record(model_name, y_train_pred, y_test_pred):
    global result_MAE, result_RMSE

    mae_train = mean_absolute_error(y_train, y_train_pred)
    mae_test = mean_absolute_error(y_test, y_test_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

    result_MAE = pd.concat([
        result_MAE,
        pd.DataFrame([{
            'model': model_name,
            'train': mae_train,
            'test': mae_test,
        }]),
    ], ignore_index=True)

    result_RMSE = pd.concat([
        result_RMSE,
        pd.DataFrame([{
            'model': model_name,
            'train': rmse_train,
            'test': rmse_test,
        }]),
    ], ignore_index=True)


print('Функция записи результатов готова')

Функция записи результатов готова


### 5.b  Линейная регрессия

In [23]:
lr = LinearRegression()
lr.fit(X_train_poly, y_train)

y_train_lr = lr.predict(X_train_poly)
y_test_lr = lr.predict(X_test_poly)

record('linear_regression', y_train_lr, y_test_lr)
print('Линейная регрессия обучена.')

Линейная регрессия обучена.


### 5.c  Решающее дерево (random_state = 21)

In [24]:
dt = DecisionTreeRegressor(random_state=21)
dt.fit(X_train_poly, y_train)

y_train_dt = dt.predict(X_train_poly)
y_test_dt = dt.predict(X_test_poly)

record('decision_tree', y_train_dt, y_test_dt)
print('Решающее дерево обучено.')

Решающее дерево обучено.


### 5.d  Наивные модели (среднее и медиана)

In [25]:
# Наивная модель: среднее
y_mean = y_train.mean()
y_train_mean = np.full_like(y_train, y_mean, dtype=float)
y_test_mean = np.full_like(y_test, y_mean, dtype=float)
record('naive_mean', y_train_mean, y_test_mean)

# Наивная модель: медиана
y_median = np.median(y_train)
y_train_median = np.full_like(y_train, y_median, dtype=float)
y_test_median = np.full_like(y_test, y_median, dtype=float)
record('naive_median', y_train_median, y_test_median)

print(f'Наивное среднее:   ${y_mean:,.0f}')
print(f'Наивная медиана:   ${y_median:,.0f}')

Наивное среднее:   $3,540
Наивная медиана:   $3,150


### 5.e  Сравнение

In [26]:
print('=== MAE ===')
display(result_MAE)
print()
print('=== RMSE ===')
display(result_RMSE)

=== MAE ===


,model,train,test
0,linear_regression,756.726509,759.829895
1,decision_tree,756.704419,754.046733
2,naive_mean,1140.445303,1136.621987
3,naive_median,1087.459008,1081.216618



=== RMSE ===


,model,train,test
0,linear_regression,1079.066347,1247.298814
1,decision_tree,1078.967775,1074.052189
2,naive_mean,1598.460491,1594.392461
3,naive_median,1645.459174,1639.336503


In [27]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

result_MAE.set_index('model')[['train', 'test']].plot.bar(
    ax=axes[0], edgecolor='white'
)
axes[0].set_title('MAE по моделям')
axes[0].set_ylabel('MAE ($)')
axes[0].tick_params(axis='x', rotation=20)

result_RMSE.set_index('model')[['train', 'test']].plot.bar(
    ax=axes[1], edgecolor='white'
)
axes[1].set_title('RMSE по моделям')
axes[1].set_ylabel('RMSE ($)')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

### Какая модель лучше?



Показатели могут несколько отличаться в зависимости от версии sklearn.



- **Решающее дерево** демонстрирует наименьшую MAE на тесте и наименьший RMSE на тесте, при этом хорошо обобщает данные.

- **Линейная регрессия** с полиномиальными признаками степени 10 имеет низкую MAE на тесте, но очень высокий RMSE из-за экстраполяции на выбросах.

- **Наивные модели** (среднее / медиана) имеют бо́льшие ошибки, чем обе ML-модели.



**Вывод:** **решающее дерево** (random_state = 21) — лучшая модель для этой задачи.